# Serving tier on a Colab T4: engine, parity gate, latency sweep

Runs `docs/AGENT_BRIEF.md` Phase 4. Two processes share one card: the LLM
serving engine, and the pipeline that talks to it over HTTP.

**Runtime → Change runtime type → T4 GPU** before starting.

Order matters and is not arbitrary:

1. install both repos
2. start the engine on **Qwen3-4B** with a pool sized for one stream, and
   **wait for `/ready`** — not `/health`, which answers alive all through
   CUDA-graph capture and Triton JIT
3. `scripts/engine_parity.py` — the gate. Until it exits 0, any latency this
   engine produces is the latency of a model that answers differently
4. `scripts/latency_ab.py` — four interleaved arms
5. apply the rules already written in `docs/EXPERIMENTS.md`, without
   re-deriving them from the numbers you just saw

Of a 4435.5 ms p50 turn, the LLM was 3324 ms and ASR was 342 ms *and off the
critical path*. This notebook is aimed at the 3324 ms.

The model is Qwen3-4B. Two things to hold in mind while reading the
results: this project's own T4 bake-off measured 4B at **3553 ms**
first-sentence p50 against 0.6B's **1744 ms** on the explicit runner,
so the engine has to recover more than 2× for the swap to pay on
latency; and speculative decoding, which would most plausibly cover
4B's decode cost, needs two GPUs and is unavailable here. That is why
the sweep runs both checkpoints rather than only the new default.

## 1 · GPU and both repositories

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# The pipeline. Use your own remote if you have push access.
%cd /content
!git clone -q https://github.com/Vaibhav7711/indic-voice-pipeline.git || true
%cd /content/indic-voice-pipeline
!git pull -q --ff-only || true
!pip install -q -e ".[audio]" 2>&1 | tail -2
!git log --oneline -1

In [ ]:
# The serving engine, beside it. Its extras: torch/transformers/triton must
# match the runtime's CUDA, which on Colab they already do.
%cd /content
!git clone -q https://github.com/Vaibhav7711/full-inference-engine.git || true
%cd /content/full-inference-engine
!git pull -q --ff-only || true
!pip install -q -e ".[server]" 2>&1 | tail -3
!git log --oneline -1

### What this device will actually serve with

`check_hooks.py --backends-only` prints the checkpoint's geometry, what got
fused, and every attention backend with `ok`/`no` **and the reason**. On a T4
(sm_75) expect FlashAttention to be unavailable — its wheels need sm_80+ —
and the tiled Triton prefill to be refused because `tl.dot` emits no
`mma.sync` there. Both are measured negative results, not failures. The
engine's per-architecture policy picks the settings an A/B on *this*
architecture chose.

In [ ]:
%cd /content/full-inference-engine
!python scripts/check_hooks.py --backends-only

## 2 · Start the engine and wait for warmup

`nohup` so the cell returns; `--wait-only` then blocks until `/ready` is 200.
Warmup is a startup cost — graph capture across every forward shape plus
Triton JIT — and it must not land inside the first measured turn.

If `--wait-only` times out or reports the process gone, read `engine.log`:
the engine refuses geometry its paged kernels cannot serve *at load, with the
reason*, and that message is the answer.

In [ ]:
import os, subprocess
os.chdir('/content/indic-voice-pipeline')

# Qwen3-4B through this repo's factory, which sizes the pool for one stream:
# 512 x 16 = 8192 KV tokens at 144 KiB/token = 1.125 GiB, against 2.25 GiB for
# the engine's 1024-block default. Weights are ~7.5 GiB, so with Whisper in
# the other process the card holds ~10.7 GiB before graphs and activations.
env = dict(os.environ,
           PYTHONPATH='/content/full-inference-engine:/content/indic-voice-pipeline',
           LLM_SERVER_MODEL='Qwen/Qwen3-4B',
           LLM_SERVER_NUM_BLOCKS='512',
           LLM_SERVER_BLOCK_SIZE='16',
           LLM_SERVER_MAX_ACTIVE='2',
           LLM_SERVER_GRAPH_BUCKETS='1,2',
           LLM_SERVER_DTYPE='float16')

from scripts.llm_server_app import describe, resolve_config
print('memory before allocating:', describe(resolve_config(env)))

log = open('/content/engine.log', 'w')
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'scripts.llm_server_app:create', '--factory',
     '--host', '127.0.0.1', '--port', '8000'],
    stdout=log, stderr=subprocess.STDOUT, env=env)
print('pid', server.pid)

In [ ]:
# 4B is ~8 GB of weights to download and load, then graph capture and Triton
# JIT on top, so give warmup room. /health would answer alive throughout.
!python scripts/serve_llm.py --wait-only --ready-timeout 1800
!tail -8 /content/engine.log

In [ ]:
# What the engine says about itself once it is up.
!curl -s localhost:8000/v1/models && echo
!curl -s localhost:8000/ready && echo

## 3 · The parity gate

Both decoders are greedy: the engine documents `temperature = 0` as greedy
and taking precedence over every other knob, and the explicit runner is
greedy. Same weights, same prompt, same text — or the faster engine is not
serving the same model.

Gates on the first 24 characters, which is what a listener hears before the
first unit reaches TTS. Late divergence is fp16 non-associativity and passes
by design; **early** divergence means prefill differs and is a real defect.

Exit status 0 or the rest of this notebook does not count.

In [ ]:
# --dtype must match what the server loaded. A bf16 reference against an fp16
# server is two different sets of numerics, and two greedy decoders over
# different numerics diverge for reasons that say nothing about the engine.
!python scripts/engine_parity.py \
    --llm-model Qwen/Qwen3-4B \
    --dtype float16 \
    --llm-base-url http://127.0.0.1:8000/v1 \
    --note "colab t4, qwen3-4b, scripts.llm_server_app 512x16"
print('---'); import json
r = json.load(open('results/engine_parity/parity.json'))
print(json.dumps(r['summary'], indent=2))

# Read corrupted_prompts FIRST. While it is non-zero the agreement figures
# measure the corruption, not decoder agreement: Qwen's byte-level BPE splits
# a 3-byte Devanagari character across two tokens, and a server that streams
# the diff of its decoded-so-far text emits U+FFFD for the partial character
# and cannot retract it. See docs/ENGINE_BUG_UTF8_STREAMING.md for the patch.
# A larger model does not fix it; English does not show it.
if r['summary']['corrupted_prompts']:
    print(f"\nSTOP: {r['summary']['corrupted_prompts']} corrupted responses, "
          f"{r['summary']['served_replacement_chars']} U+FFFD characters.")
    print("Fix the engine's _stream before reading anything below as latency.")
for item in r['results']:
    if item['comparison']['corruption']:
        print('CORRUPT:', item['prompt']); print('  srv:', item['served'][:120])
    elif not item['comparison']['agreed']:
        print('DIVERGED:', item['prompt']); print('  ref:', item['reference'][:120])
        print('  srv:', item['served'][:120])

## 4 · The latency sweep

Four arms, interleaved one turn each per round, sharing one set of loaded
weights. `--rounds 8` because the history-budget effect grows with session
length — at turn 1 there is no history to trim, so a short run understates
it.

`--tts mms` keeps a network voice's bad minute out of the measurement. Drop
to `--tts edge` only if VRAM is tight.

In [ ]:
# Both checkpoints, because an A/B that changes the model and the engine at
# once cannot attribute the difference to either. The explicit 0.6B arm is the
# baseline the pre-registered rule names.
#
# --llm-model sets the LOCAL explicit arm. The served arm answers from
# whatever the server loaded (4B); the parity run above is what confirms which.
!python scripts/latency_ab.py --rounds 8 \
    --llm-engine explicit --llm-engine http \
    --llm-model Qwen/Qwen3-0.6B \
    --llm-base-url http://127.0.0.1:8000/v1 \
    --tts mms \
    --arm baseline \
    --arm served4b:llm_engine=http \
    --arm history200:history_tokens=200 \
    --arm units30:unit_chars=30 \
    --note "colab t4; baseline=explicit 0.6B, served4b=engine Qwen3-4B"

In [ ]:
import json
summary = json.load(open('results/latency_ab/summary.json'))
PRIMARY = summary['primary_metric']       # committed transcript -> first audio
print('primary metric:', PRIMARY)

def fmt(v):
    return f'{v:8.1f}' if isinstance(v, (int, float)) else '     n/a'

for name, block in summary['arms'].items():
    print(f"{name:<14} n={block['turns']:<3} "
          f"p50={fmt(block[PRIMARY]['p50'])}  "
          f"p90={fmt(block[PRIMARY]['p90'])}  "
          f"first_token={fmt(block['final_transcript_to_first_llm_token_ms']['p50'])}  "
          f"to_unit={fmt(block['first_token_to_first_unit_ms']['p50'])}")

# response_latency_ms is n/a here by construction: these turns carry no speech,
# so the endpoint-to-final segment does not exist and is not made up. The
# measured 660.8 ms floor sits under any perceived-latency figure.
base = summary['arms']['baseline'][PRIMARY]['p50']
served = summary['arms']['served4b'][PRIMARY]['p50']
if isinstance(base, (int, float)) and isinstance(served, (int, float)) and served:
    print(f"\nserved 4B vs explicit 0.6B: {base / served:.4f}x on {PRIMARY} p50")
    print("pre-registered rule: keep 4B only if this is at or above 1.0. The T4")
    print("bake-off measured 4B at 3553 ms first-sentence p50 against 0.6B at")
    print("1744 ms on the explicit runner, so the engine must recover over 2x.")
else:
    print("\nno ratio: an arm did not reach audio. That is a failure to "
          "investigate, not a missing number to fill in.")

## 5 · The human conditions

Two arms cannot be closed by a number, and the pre-registered rules say so.

**`history200`** — does a 200-token budget still hold a conversation? Ask a
follow-up that depends on the previous turn. If the assistant answers as
though the exchange never happened, the latency it bought is not a win for a
dialogue agent.

**`units30`** — listen. A shorter first unit buys silence-to-speech by
cutting the sentence in a slightly odder place.

In [ ]:
# Text in, audio out: this checks dialogue memory, not ASR, so feeding fixed
# text is what makes the two history budgets comparable at all. Same builder
# the A/B uses, so the arm here is the arm that was measured.
import os, sys
sys.path.insert(0, '/content/indic-voice-pipeline')
os.chdir('/content/indic-voice-pipeline')

from agent.audio import DecodingBufferSink
from llm.engines import build_llm
from scripts.latency_ab import build_arm
from tts.local import MmsTtsSynthesizer

generator, tokenizer, info = build_llm(
    'http', model='Qwen/Qwen3-4B', base_url='http://127.0.0.1:8000/v1')
print(info)
synth = MmsTtsSynthesizer('hi'); synth.warm_up()

# A follow-up whose meaning depends on the previous turn. If the shorter
# budget has dropped the antecedent, the second answer will not resolve
# "वहाँ" and the third will not resolve "उसका".
FOLLOW_UPS = ["भारत की राजधानी क्या है?",
              "वहाँ का मौसम कैसा रहता है?",
              "उसका मतलब क्या है?"]

for tokens in (800, 200):
    turn = build_arm({'history_tokens': tokens},
                     base={'history_tokens': 800, 'history_turns': 6,
                           'unit_chars': 60, 'llm_engine': 'http'},
                     generators={'http': (generator, tokenizer)}, synth=synth,
                     sink_factory=lambda: DecodingBufferSink(synth.format))
    print(f"\n=== max_history_tokens={tokens} ===")
    for question in FOLLOW_UPS:
        result = turn.run(question)
        latency = result.as_dict().get('response_latency_ms')
        shown = f"{latency:.0f} ms" if isinstance(latency, (int, float)) else "n/a"
        print(f"  Q: {question}\n  A: {result.response}  [{shown}]")

print("\nRead the 200-token answers: did the follow-ups still resolve? That "
      "judgement is yours, not a threshold's.")

## 6 · Record it

Write the numbers into the ledger under the pre-registered section, apply the
rules literally, and commit. An arm that was measured but not decided, or
decided but not recorded, is the same as one that never ran.

In [ ]:
!git -C /content/indic-voice-pipeline add results/engine_parity results/latency_ab
!git -C /content/indic-voice-pipeline status --short

In [ ]:
# Stop the engine. SIGINT drains in-flight requests rather than killing them.
import signal
server.send_signal(signal.SIGINT)
print('exit', server.wait(timeout=60))
!tail -3 /content/engine.log